In [1]:
import torch
torch.hub.set_dir('/kaggle/working/torch_hub')

In [2]:
"""
Segmentation Training Script
Converted from train_mask.ipynb
Trains a segmentation head on top of DINOv2 backbone
"""

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image
import cv2
import os
import torchvision
from tqdm import tqdm
from torchvision.transforms import InterpolationMode

# Set matplotlib to non-interactive backend
plt.switch_backend('Agg')

In [3]:
# ============================================================================
# Hyperparameters & Configuration
# ============================================================================

# Training Settings
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
N_EPOCHS = 10
BACKBONE_SIZE = "small"  # Options: "small", "base", "large", "giant"

# Image Dimensions (Must be multiples of 14 for DINOv2)
W = int((960 // 14) * 14)
H = int((540 // 14) * 14)

# Paths
SCRIPT_DIR = "/content/working" if not "/kaggle/working" else "/kaggle/working"
OUTPUT_DIR ="/kaggle/working/train_stats"
# Loss Weighting (CombinedLoss alpha)
LOSS_ALPHA = 0.5

print(f"Configuration Loaded:\nResolution: {W}x{H}\nBatch Size: {BATCH_SIZE}\nLearning Rate: {LEARNING_RATE}\nEpochs: {N_EPOCHS}")

Configuration Loaded:
Resolution: 952x532
Batch Size: 16
Learning Rate: 0.0001
Epochs: 10


In [4]:

# ============================================================================
# Utility Functions
# ============================================================================

def save_image(img, filename):
    """Save an image tensor to file after denormalizing."""
    img = np.array(img)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = np.moveaxis(img, 0, -1)
    img = (img * std + mean) * 255
    cv2.imwrite(filename, img[:, :, ::-1])


# ============================================================================
# Mask Conversion
# ============================================================================

# Mapping from raw pixel values to new class IDs
value_map = {
    0: 0,        # background
    100: 1,      # Trees
    200: 2,      # Lush Bushes
    300: 3,      # Dry Grass
    500: 4,      # Dry Bushes
    550: 5,      # Ground Clutter
    700: 6,      # Logs
    800: 7,      # Rocks
    7100: 8,     # Landscape
    10000: 9     # Sky
}
n_classes = len(value_map)


def convert_mask(mask):
    """Convert raw mask values to class IDs."""
    arr = np.array(mask)
    new_arr = np.zeros_like(arr, dtype=np.uint8)
    for raw_value, new_value in value_map.items():
        new_arr[arr == raw_value] = new_value
    return Image.fromarray(new_arr)

In [5]:

# ============================================================================
# Dataset
# ============================================================================

class MaskDataset(Dataset):
    def __init__(self, data_dir, transform=None, mask_transform=None):
        self.image_dir = os.path.join(data_dir, 'Color_Images')
        self.masks_dir = os.path.join(data_dir, 'Segmentation')
        self.transform = transform
        self.mask_transform = mask_transform
        self.data_ids = os.listdir(self.image_dir)

    def __len__(self):
        return len(self.data_ids)

    def __getitem__(self, idx):
        data_id = self.data_ids[idx]
        img_path = os.path.join(self.image_dir, data_id)
        # Both color images and masks are .png files with same name
        mask_path = os.path.join(self.masks_dir, data_id)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)
        mask = convert_mask(mask)

        if self.transform:
             image = self.transform(image)
        if self.mask_transform:
             mask = self.mask_transform(mask)
        mask = torch.from_numpy(np.array(mask)).long()

        return image, mask

In [6]:
class ASPPFPNSegmentationHead(nn.Module):
    def __init__(self, in_channels, out_channels, tokenW, tokenH):
        super().__init__()
        self.H, self.W = tokenH, tokenW

        # ASPP branches with adjusted dilation rates for small feature maps
        self.aspp_1x1 = nn.Sequential(
            nn.Conv2d(in_channels, 256, 1),
            nn.BatchNorm2d(256),
            nn.GELU()
        )
        self.aspp_d2 = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=2, dilation=2),
            nn.BatchNorm2d(256),
            nn.GELU()
        )
        self.aspp_d4 = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=4, dilation=4),
            nn.BatchNorm2d(256),
            nn.GELU()
        )
        self.aspp_d8 = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=8, dilation=8),
            nn.BatchNorm2d(256),
            nn.GELU()
        )
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, 256, 1),
            nn.GELU()
        )

        # Fuse all 5 ASPP branches
        self.aspp_fuse = nn.Sequential(
            nn.Conv2d(256 * 5, 256, 1),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.Dropout2d(0.1)
        )

        # FPN progressive upsampling
        self.up1 = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU()
        )
        self.up2 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU()
        )

        self.classifier = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        B, N, C = x.shape
        x = x.reshape(B, self.H, self.W, C).permute(0, 3, 1, 2)

        # ASPP - all branches in parallel
        b1 = self.aspp_1x1(x)
        b2 = self.aspp_d2(x)
        b3 = self.aspp_d4(x)
        b4 = self.aspp_d8(x)
        b5 = F.interpolate(self.gap(x), size=x.shape[2:], mode='bilinear', align_corners=False)

        # Fuse
        x = self.aspp_fuse(torch.cat([b1, b2, b3, b4, b5], dim=1))

        # FPN progressive upsampling
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = self.up1(x)

        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = self.up2(x)

        return self.classifier(x)

In [7]:
def compute_iou(pred, target, num_classes=10, ignore_index=255):
    pred = torch.argmax(pred, dim=1)
    pred, target = pred.view(-1), target.view(-1)
    iou_per_class = []
    for class_id in range(num_classes):
        if class_id == ignore_index: continue
        pred_inds = pred == class_id
        target_inds = target == class_id
        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()
        if union == 0: iou_per_class.append(float('nan'))
        else: iou_per_class.append((intersection / union).cpu().numpy())
    return np.nanmean(iou_per_class)

def compute_dice(pred, target, num_classes=10, smooth=1e-6):
    pred = torch.argmax(pred, dim=1)
    pred, target = pred.view(-1), target.view(-1)
    dice_per_class = []
    for class_id in range(num_classes):
        pred_inds = pred == class_id
        target_inds = target == class_id
        intersection = (pred_inds & target_inds).sum().float()
        dice_score = (2. * intersection + smooth) / (pred_inds.sum().float() + target_inds.sum().float() + smooth)
        dice_per_class.append(dice_score.cpu().numpy())
    return np.mean(dice_per_class)

def compute_pixel_accuracy(pred, target):
    pred_classes = torch.argmax(pred, dim=1)
    return (pred_classes == target).float().mean().cpu().numpy()

def evaluate_metrics(model, backbone, data_loader, device, loss_fct, num_classes=10, show_progress=True):
    iou_scores, dice_scores, pixel_accuracies, losses = [], [], [], []
    model.eval()
    backbone.eval()
    loader = tqdm(data_loader, desc="Evaluating", leave=False, unit="batch") if show_progress else data_loader
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            output = backbone.forward_features(imgs)["x_norm_patchtokens"]
            logits = model(output)
            outputs = F.interpolate(logits, size=imgs.shape[2:], mode="bilinear", align_corners=False)
            labels = F.interpolate(labels.unsqueeze(1).float(), size=outputs.shape[2:], mode="nearest").squeeze(1).long()

            loss = loss_fct(outputs, labels)
            losses.append(loss.item())
            iou_scores.append(compute_iou(outputs, labels, num_classes=num_classes))
            dice_scores.append(compute_dice(outputs, labels, num_classes=num_classes))
            pixel_accuracies.append(compute_pixel_accuracy(outputs, labels))
    model.train()
    return np.mean(iou_scores), np.mean(dice_scores), np.mean(pixel_accuracies), np.mean(losses)

In [8]:

# ============================================================================
# Plotting Functions
# ============================================================================

def save_training_plots(history, output_dir):
    """Save all training metric plots to files."""
    os.makedirs(output_dir, exist_ok=True)

    # Plot 1: Loss curves
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='train')
    plt.plot(history['val_loss'], label='val')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history['train_pixel_acc'], label='train')
    plt.plot(history['val_pixel_acc'], label='val')
    plt.title('Pixel Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_curves.png'))
    plt.close()
    print(f"Saved training curves to '{output_dir}/training_curves.png'")

    # Plot 2: IoU curves
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_iou'], label='Train IoU')
    plt.title('Train IoU vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history['val_iou'], label='Val IoU')
    plt.title('Validation IoU vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'iou_curves.png'))
    plt.close()
    print(f"Saved IoU curves to '{output_dir}/iou_curves.png'")

    # Plot 3: Dice curves
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_dice'], label='Train Dice')
    plt.title('Train Dice vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Score')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history['val_dice'], label='Val Dice')
    plt.title('Validation Dice vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Score')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'dice_curves.png'))
    plt.close()
    print(f"Saved Dice curves to '{output_dir}/dice_curves.png'")

    # Plot 4: Combined metrics plot
    plt.figure(figsize=(12, 10))

    plt.subplot(2, 2, 1)
    plt.plot(history['train_loss'], label='train')
    plt.plot(history['val_loss'], label='val')
    plt.title('Loss vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 2, 2)
    plt.plot(history['train_iou'], label='train')
    plt.plot(history['val_iou'], label='val')
    plt.title('IoU vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 2, 3)
    plt.plot(history['train_dice'], label='train')
    plt.plot(history['val_dice'], label='val')
    plt.title('Dice Score vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Score')
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 2, 4)
    plt.plot(history['train_pixel_acc'], label='train')
    plt.plot(history['val_pixel_acc'], label='val')
    plt.title('Pixel Accuracy vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Pixel Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'all_metrics_curves.png'))
    plt.close()
    print(f"Saved combined metrics curves to '{output_dir}/all_metrics_curves.png'")


def save_history_to_file(history, output_dir):
    """Save training history to a text file."""
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, 'evaluation_metrics.txt')

    with open(filepath, 'w') as f:
        f.write("TRAINING RESULTS\n")
        f.write("=" * 50 + "\n\n")

        f.write("Final Metrics:\n")
        f.write(f"  Final Train Loss:     {history['train_loss'][-1]:.4f}\n")
        f.write(f"  Final Val Loss:       {history['val_loss'][-1]:.4f}\n")
        f.write(f"  Final Train IoU:      {history['train_iou'][-1]:.4f}\n")
        f.write(f"  Final Val IoU:        {history['val_iou'][-1]:.4f}\n")
        f.write(f"  Final Train Dice:     {history['train_dice'][-1]:.4f}\n")
        f.write(f"  Final Val Dice:       {history['val_dice'][-1]:.4f}\n")
        f.write(f"  Final Train Accuracy: {history['train_pixel_acc'][-1]:.4f}\n")
        f.write(f"  Final Val Accuracy:   {history['val_pixel_acc'][-1]:.4f}\n")
        f.write("=" * 50 + "\n\n")

        f.write("Best Results:\n")
        f.write(f"  Best Val IoU:      {max(history['val_iou']):.4f} (Epoch {np.argmax(history['val_iou']) + 1})\n")
        f.write(f"  Best Val Dice:     {max(history['val_dice']):.4f} (Epoch {np.argmax(history['val_dice']) + 1})\n")
        f.write(f"  Best Val Accuracy: {max(history['val_pixel_acc']):.4f} (Epoch {np.argmax(history['val_pixel_acc']) + 1})\n")
        f.write(f"  Lowest Val Loss:   {min(history['val_loss']):.4f} (Epoch {np.argmin(history['val_loss']) + 1})\n")
        f.write("=" * 50 + "\n\n")

        f.write("Per-Epoch History:\n")
        f.write("-" * 100 + "\n")
        headers = ['Epoch', 'Train Loss', 'Val Loss', 'Train IoU', 'Val IoU',
                   'Train Dice', 'Val Dice', 'Train Acc', 'Val Acc']
        f.write("{:<8} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12}\n".format(*headers))
        f.write("-" * 100 + "\n")

        n_epochs = len(history['train_loss'])
        for i in range(n_epochs):
            f.write("{:<8} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f}\n".format(
                i + 1,
                history['train_loss'][i],
                history['val_loss'][i],
                history['train_iou'][i],
                history['val_iou'][i],
                history['train_dice'][i],
                history['val_dice'][i],
                history['train_pixel_acc'][i],
                history['val_pixel_acc'][i]
            ))

    print(f"Saved evaluation metrics to {filepath}")

In [9]:
def save_checkpoint(classifier, optimizer, scheduler, epoch, history, output_dir):
    torch.save({
        'epoch': epoch,
        'classifier_state_dict': classifier.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history,
    }, os.path.join(output_dir, 'checkpoint.pth'))
    print(f"Checkpoint saved at epoch {epoch + 1}")

def load_checkpoint(classifier, optimizer, scheduler, output_dir):
    path = os.path.join(output_dir, 'checkpoint.pth')
    if not os.path.exists(path):
        print("No checkpoint found, starting fresh.")
        return 0, {k: [] for k in ['train_loss','val_loss','train_iou','val_iou','train_dice','val_dice','train_pixel_acc','val_pixel_acc']}
    ckpt = torch.load(path)
    classifier.load_state_dict(ckpt['classifier_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    print(f"Resumed from epoch {ckpt['epoch'] + 1}")
    return ckpt['epoch'] + 1, ckpt['history']

def save_model_weights(classifier, backbone_model, output_dir):
    torch.save(classifier.state_dict(), os.path.join(output_dir, 'classifier_weights.pth'))
    torch.save(backbone_model.state_dict(), os.path.join(output_dir, 'backbone_weights.pth'))
    print(f"Model weights saved to {output_dir}")

In [10]:
class OptimizedCombinedLoss(nn.Module):
    def __init__(self, num_classes, alpha=0.3, gamma=3.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.num_classes = num_classes

    def focal_loss(self, pred, target):
        ce_loss = F.cross_entropy(pred, target, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss

    def dice_loss(self, pred, target):
        smooth = 1e-6
        pred = F.softmax(pred, dim=1)
        target_one_hot = F.one_hot(target.clamp(min=0, max=self.num_classes-1), num_classes=self.num_classes).permute(0, 3, 1, 2).float()
        intersection = (pred * target_one_hot).sum(dim=(0, 2, 3))
        cardinality = pred.sum(dim=(0, 2, 3)) + target_one_hot.sum(dim=(0, 2, 3))
        dice = (2. * intersection + smooth) / (cardinality + smooth)
        return 1 - dice.mean()

    def forward(self, pred, target):
        return self.alpha * self.focal_loss(pred, target) + (1 - self.alpha) * self.dice_loss(pred, target)

In [11]:
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    script_dir, output_dir = SCRIPT_DIR, os.path.join(SCRIPT_DIR, 'train_stats')
    os.makedirs(output_dir, exist_ok=True)

    transform = transforms.Compose([transforms.Resize((H, W)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    transform_train = transforms.Compose([
        transforms.Resize((H, W)),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    mask_transform = transforms.Compose([transforms.Resize((H, W), interpolation=InterpolationMode.NEAREST)])

    base_path = "/kaggle/input/datasets/divyanshsharma23/semantic-segmentation/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"
    train_loader = DataLoader(MaskDataset(os.path.join(base_path, 'train'), transform_train, mask_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(MaskDataset(os.path.join(base_path, 'val'), transform, mask_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    backbone_archs = {"small": "vits14", "base": "vitb14_reg", "large": "vitl14_reg", "giant": "vitg14_reg"}
    backbone_model = torch.hub.load("facebookresearch/dinov2", f"dinov2_{backbone_archs[BACKBONE_SIZE]}", trust_repo=True).to(device)
    for param in backbone_model.parameters(): param.requires_grad = False
    for param in backbone_model.blocks[-2:].parameters(): param.requires_grad = True
    for param in backbone_model.norm.parameters(): param.requires_grad = True

    n_embedding = backbone_model.embed_dim
    classifier = ASPPFPNSegmentationHead(in_channels=n_embedding, out_channels=n_classes, tokenW=W//14, tokenH=H//14).to(device)
    loss_fct = OptimizedCombinedLoss(num_classes=n_classes, alpha=0.3)
    optimizer = torch.optim.AdamW([{'params': classifier.parameters(), 'lr': LEARNING_RATE}, {'params': filter(lambda p: p.requires_grad, backbone_model.parameters()), 'lr': LEARNING_RATE * 0.1}], weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-4, steps_per_epoch=len(train_loader), epochs=N_EPOCHS, pct_start=0.3, div_factor=10, final_div_factor=100)

    start_epoch, history = load_checkpoint(classifier, optimizer, scheduler, output_dir)

    for epoch in range(start_epoch, N_EPOCHS):
        classifier.train(); backbone_model.train()
        epoch_losses = []
        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS}"):
            imgs, labels = imgs.to(device), labels.to(device)
            output = backbone_model.forward_features(imgs)["x_norm_patchtokens"]
            outputs = F.interpolate(classifier(output), size=imgs.shape[2:], mode="bilinear", align_corners=False)
            labels_resized = F.interpolate(labels.unsqueeze(1).float(), size=outputs.shape[2:], mode="nearest").squeeze(1).long()

            loss = loss_fct(outputs, labels_resized)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(classifier.parameters(), 1.0)
            optimizer.step(); optimizer.zero_grad(); scheduler.step()
            epoch_losses.append(loss.item())

        t_iou, t_dice, t_acc, _ = evaluate_metrics(classifier, backbone_model, train_loader, device, loss_fct, n_classes, False)
        v_iou, v_dice, v_acc, v_loss = evaluate_metrics(classifier, backbone_model, val_loader, device, loss_fct, n_classes, False)

        history['train_loss'].append(np.mean(epoch_losses)); history['val_loss'].append(v_loss)
        history['train_iou'].append(t_iou); history['val_iou'].append(v_iou)
        history['train_dice'].append(t_dice); history['val_dice'].append(v_dice)
        history['train_pixel_acc'].append(t_acc); history['val_pixel_acc'].append(v_acc)

        print(f"\nEpoch [{epoch+1}/{N_EPOCHS}] - Train Loss: {history['train_loss'][-1]:.4f}, Val Loss: {v_loss:.4f}")
        print(f"Val IoU: {v_iou:.4f}, Val Dice: {v_dice:.4f}, Val Acc: {v_acc:.4f}\n")
        save_checkpoint(classifier, optimizer, scheduler, epoch, history, output_dir)  # ✅ fixed

    save_training_plots(history, output_dir)
    save_history_to_file(history, output_dir)
    save_model_weights(classifier, backbone_model, output_dir)  # ✅ correct - saves both at end
    print("Training complete!")

In [12]:
if __name__ == "__main__":
    main()

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /kaggle/working/torch_hub/main.zip


/kaggle/working/torch_hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/torch_hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/torch_hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /kaggle/working/torch_hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 284MB/s]


No checkpoint found, starting fresh.


...


Epoch [8/10] - Train Loss: 0.2815, Val Loss: 0.2775
Val IoU: 0.6055, Val Dice: 0.7353, Val Acc: 0.8521

Checkpoint saved at epoch 8


Epoch 9/10: 100%|██████████| 179/179 [08:10<00:00,  2.74s/it]



Epoch [9/10] - Train Loss: 0.2752, Val Loss: 0.2739
Val IoU: 0.6096, Val Dice: 0.7390, Val Acc: 0.8529

Checkpoint saved at epoch 9


Epoch 10/10: 100%|██████████| 179/179 [08:10<00:00,  2.74s/it]



Epoch [10/10] - Train Loss: 0.2734, Val Loss: 0.2735
Val IoU: 0.6100, Val Dice: 0.7392, Val Acc: 0.8535

Checkpoint saved at epoch 10
Saved training curves to '/kaggle/working/train_stats/training_curves.png'
Saved IoU curves to '/kaggle/working/train_stats/iou_curves.png'
Saved Dice curves to '/kaggle/working/train_stats/dice_curves.png'
Saved combined metrics curves to '/kaggle/working/train_stats/all_metrics_curves.png'
Saved evaluation metrics to /kaggle/working/train_stats/evaluation_metrics.txt
Model weights saved to /kaggle/working/train_stats
Training complete!
